# 12장. 평가

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


드디어 **test 세트**로 최종 성능을 잽니다. test는 학습에도, 최고 모델 고르기에도 쓰지 않은 데이터입니다. 5장에서 맛보기 test는 **K002(정상)와 KA04(고장)를 통째로** 빼 두었습니다. 즉 모델이 처음 보는 베어링입니다.

## 12.1 평가 지표

| 지표 | 계산 | 의미 |
|---|---|---|
| 정확도 | 맞힌 수 ÷ 전체 | 전체적으로 얼마나 맞히나 |
| **오경보율** | 정상인데 고장이라 한 수 ÷ 실제 정상 | 쓸데없는 정비 출동 |
| **놓침률** | 고장인데 정상이라 한 수 ÷ 실제 고장 | **사고로 이어짐 (가장 위험)** |

정확도 하나만 보면 안 되고, **베어링별**, **운전 조건별**로 나눠 봐야 어디서 틀리는지 보입니다.

In [2]:
import os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"                 # 인터넷 없이 로컬 모델 폴더만 사용
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"    # Mac: MPS가 모르는 연산은 CPU로
warnings.filterwarnings("ignore")
import torch
from ch09_train_teacher import pick_device
DEVICE = pick_device()
print("장치:", DEVICE)

장치: mps


In [3]:
%run code/ch12_evaluate.py

test: 윈도우 113개, 베어링 ['K002', 'KA04']


Loading weights from local directory



[Teacher] 정확도 48/113 = 42.5%  혼동행렬 [[48, 9], [56, 0]]
  오경보율 15.8%  놓침률 100.0%
  K002: 48/57 (84%)   N09_M07_F10:14/14  N15_M01_F10:7/15  N15_M07_F04:14/14  N15_M07_F10:13/14
  KA04: 0/56 (0%)   N09_M07_F10:0/14  N15_M01_F10:0/14  N15_M07_F04:0/14  N15_M07_F10:0/14


Loading weights from local directory



[Student] 정확도 47/113 = 41.6%  혼동행렬 [[47, 10], [56, 0]]
  오경보율 17.5%  놓침률 100.0%
  K002: 47/57 (82%)   N09_M07_F10:14/14  N15_M01_F10:7/15  N15_M07_F04:14/14  N15_M07_F10:12/14
  KA04: 0/56 (0%)   N09_M07_F10:0/14  N15_M01_F10:0/14  N15_M07_F04:0/14  N15_M07_F10:0/14


## 12.2 결과 읽기: validation 100%인데 test 40%대

9장과 11장에서 validation은 28/28(100%)이었습니다. 그런데 처음 보는 베어링에서는:

- **K002(정상)**: 대부분 맞힘. 다만 `N15_M01_F10`(저부하) 조건에서 절반을 고장으로 오판
- **KA04(고장)**: **전부 놓침** (놓침률 100%)

학습한 고장 베어링은 KA03 하나뿐이었습니다. 모델은 "고장의 일반적 특징"이 아니라 **"KA03의 특징"**을 배운 것으로 보입니다. 그래서 같은 외륜 손상이라도 다른 베어링인 KA04를 못 알아봅니다.

## 12.3 실제 프로젝트(v3)에서도 같은 일이 일어났다

실제 프로젝트는 정상 3개, 고장 3개(KA03, KA04, KI03) 베어링으로 학습하고 K004, KB24, KB27로 test했습니다. 같은 평가 코드로 실제 v3 Student를 평가해 봅니다.

In [4]:
import ch12_evaluate as ch12, json
from ch06_dataset import WindowDataset
from ch10_checkpoint import load_full

v3_split = json.loads((ROOT / "data/splits/bearing_holdout_v3.json").read_text())
v3_test = WindowDataset([ROOT / f for f in v3_split["test"]])
v3_student = load_full(ROOT / "models/MOMENT-1-small", ROOT / "outputs/runs/v3/student/student_kd_best.pt", teacher=False)
v3_logits = ch12.predict(v3_student, v3_test, DEVICE)
_ = ch12.report("v3 Student", v3_logits, v3_test)

Loading weights from local directory



[v3 Student] 정확도 1086/1681 = 64.6%  혼동행렬 [[525, 35], [560, 561]]
  오경보율 6.2%  놓침률 50.0%
  K004: 525/560 (94%)   N09_M07_F10:135/140  N15_M01_F10:121/140  N15_M07_F04:140/140  N15_M07_F10:129/140
  KB24: 0/560 (0%)   N09_M07_F10:0/140  N15_M01_F10:0/140  N15_M07_F04:0/140  N15_M07_F10:0/140
  KB27: 561/561 (100%)   N09_M07_F10:140/140  N15_M01_F10:141/141  N15_M07_F04:140/140  N15_M07_F10:140/140


| test 베어링 | 결과 | 해석 |
|---|---|---|
| K004 (정상) | 94% | 처음 보는 정상 베어링도 대체로 정상으로 본다 |
| KB24 (복합 고장) | **0%** | 한 윈도우도 못 잡음 |
| KB27 (복합 고장) | 100% | 완벽 |

같은 복합 손상인데 KB24는 0%, KB27은 100%입니다. 모델이 고장을 이해했다기보다, **학습한 베어링들과 신호가 닮았는지**에 따라 결과가 갈리는 것으로 보입니다.

## 12.4 무엇을 의심해 볼 수 있나

| 가설 | 근거 | 확인 방법 |
|---|---|---|
| 학습 베어링이 너무 적다 | 고장 베어링 3개로는 다양성이 부족 | KA/KI 베어링을 더 추가 |
| validation이 너무 쉽다 | 같은 베어링의 다른 파일이라 항상 100% | validation도 베어링 단위로 분리 |
| 결함 정보가 전처리에서 사라졌다 | 4장에서 1 kHz 위를 모두 버림. 이 데이터는 2–10 kHz에 에너지가 많다 | 대역을 넓힌 전처리와 비교 (센서가 허용한다면) |
| Student가 특징을 못 바꾼다 | head만 학습 (1,026개) | LoRA 유무, KD 유무 비교 실험 |

> **이 매뉴얼의 핵심 교훈:** 모델을 만드는 것보다 **올바르게 평가하는 것**이 더 어렵고 중요합니다. validation 100%라는 숫자만 믿었다면, 특정 고장을 전혀 못 잡는 모델을 현장에 배포했을 것입니다.

## 12.5 그래도 다음 단계로

성능은 아직 부족하지만, **배포 과정 자체**(ONNX 변환 → 라즈베리파이)는 모델이 개선돼도 똑같습니다. Part 3에서는 지금 모델로 배포 과정을 익힙니다. 나중에 더 좋은 모델을 만들면 같은 과정으로 바꿔 끼우면 됩니다.

## 12장 완성 코드: `code/ch12_evaluate.py`

In [5]:
from IPython.display import Code
Code(filename=str(CODE / "ch12_evaluate.py"), language="python")

"""12장 완성 코드: test 세트로 모델을 평가하고, 베어링별·운전 조건별로 나눠 본다.

윈도우 판정 = 청크 8개의 logits 평균 → argmax (9장 window_logits)

실행 (프로젝트 루트에서):
    # 맛보기: 9·11장 결과를 5장 test(K002, KA04)로
    python docs/manual/code/ch12_evaluate.py
    # 실제 프로젝트 v3 Student를 실제 test(K004, KB24, KB27)로
    python docs/manual/code/ch12_evaluate.py --split data/splits/bearing_holdout_v3.json \
        --student outputs/runs/v3/student/student_kd_best.pt --teacher none
"""

import argparse
import json
from collections import defaultdict

import numpy as np
import torch
from torch.utils.data import DataLoader

from ch06_dataset import WindowDataset
from ch07_moment import ROOT
from ch09_train_teacher import pick_device, window_logits
from ch10_checkpoint import load_full


@torch.no_grad()
def predict(model, dataset, device, batch_size=32):
    model.to(device).eval()
    return torch.cat([window_logits(model, x.to(device)).cpu()
                      for x, _ in DataLoader(dataset, batch_size=batch_size)]).numpy()


def metrics(logits, labels):
    pred = logits.argmax(1)
    confusion = np.zeros((2, 2), dtype=int)
    np.add.at(confusion, (labels, pred), 1)
    tn, fp, fn, tp = confusion.ravel()
    return {"accuracy": float((pred == labels).mean()), "correct": int((pred == labels).sum()), "total": len(labels),
            "confusion_matrix": confusion.tolist(),
            "false_alarm_rate": float(fp / max(tn + fp, 1)),   # 정상인데 고장이라고 한 비율
            "miss_rate": float(fn / max(fn + tp, 1))}          # 고장인데 정상이라고 한 비율 (가장 위험)


def breakdown(logits, dataset):
    """베어링별, 베어링×운전 조건별 정답 수."""
    pred = logits.argmax(1)
    by_bearing, by_condition = defaultdict(lambda: [0, 0]), defaultdict(lambda: [0, 0])
    for sid, p, y in zip(dataset.sample_ids, pred, dataset.labels):
        bearing, name = sid.split("/")[0], sid.split("/")[1]
        for table, key in ((by_bearing, bearing), (by_condition, (bearing, name[:11]))):
            table[key][0] += int(p == y)
            table[key][1] += 1
    return by_bearing, by_condition


def report(name, logits, dataset):
    m = metrics(logits, dataset.labels)
    print(f"\n[{name}] 정확도 {m['correct']}/{m['total']} = {m['accuracy']:.1%}  혼동행렬 {m['confusion_matrix']}")
    print(f"  오경보율 {m['false_alarm_rate']:.1%}  놓침률 {m['miss_rate']:.1%}")
    by_bearing, by_condition = breakdown(logits, dataset)
    for bearing, (ok, n) in sorted(by_bearing.items()):
        conds = "  ".join(f"{c}:{by_condition[(bearing, c)][0]}/{by_condition[(bearing, c)][1]}"
                          for b, c in sorted(by_condition) if b == bearing)
        print(f"  {bearing}: {ok}/{n} ({ok / n:.0%})   {conds}")
    return m


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--split", default="outputs/manual/split_holdout.json")
    parser.add_argument("--teacher", default="outputs/manual/teacher/teacher_lora_best.pt", help="none이면 생략")
    parser.add_argument("--student", default="outputs/manual/student/student_kd_best.pt", help="none이면 생략")
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()

    device = pick_device(args.device)
    split = json.loads((ROOT / args.split).read_text())
    test = WindowDataset([ROOT / f for f in split["test"]])
    print(f"test: 윈도우 {len(test)}개, 베어링 {sorted({s.split('/')[0] for s in test.sample_ids})}")
    results = {}
    if args.teacher != "none":
        results["teacher"] = report("Teacher", predict(load_full(ROOT / "models/MOMENT-1-base", ROOT / args.teacher), test, device), test)
    if args.student != "none":
        results["student"] = report("Student", predict(load_full(ROOT / "models/MOMENT-1-small", ROOT / args.student, teacher=False), test, device), test)
    return results


if __name__ == "__main__":
    main()